# Day 06: Automating Your Life (Dates & Files) 📅

## 👋 Welcome Back!
Imagine you have a folder with 1,000 messy files (`data_final.txt`, `data_final_final.txt`).
You *could* rename them manually (takes 3 hours).
Or you could write a Python script to do it in 3 seconds.

Today, we learn:
1.  **`datetime`:** Handling timestamps, deadlines, and durations.
2.  **`shutil`:** Moving and copying files (The "Shell Utilities").
3.  **Automation:** Combining loops, logic, and file handling to clean up your computer.

---

## ⌚ Topic 1: The `datetime` Module
Computers count time as "Seconds passed since Jan 1, 1970". Humans use calendars.
The `datetime` module translates between the two.

### 1. Getting Current Time

In [ ]:
import datetime as dt

# Get "Now"
now = dt.datetime.now()
print(f"Current Raw Time: {now}")

# Get specific parts
print(f"Year: {now.year}")
print(f"Month: {now.month}")
print(f"Day: {now.day}")

### 2. Formatting Dates (`strftime`)
The raw format `2023-10-27 15:30:45.123456` is ugly.
We use **`strftime`** (String Format Time) to make it pretty. 

You can also remember it as "String From Time" because it converts a datetime object into a string.

* `%Y`: Year (2023)
* `%m`: Month (10)
* `%d`: Day (27)
* `%A`: Day Name (Friday)

You can get these format codes from the [Python docs](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes).

In [ ]:
# Format: "Friday, October 27"
pretty_date = now.strftime("%A, %B | %d")
print(f"Nice Date: {pretty_date}")
print(f"Nice Date type: {type(pretty_date)}")

# Format: "27/10/2023" (European style)
euro_date = now.strftime("%d/%m/%Y")
print(f"Euro Date: {euro_date}")

### 3. Parsing Dates (`strptime`)
Lot of times, You might read data from files or user input as strings. If you have a date string like "27/10/2023", you can convert it back to a datetime object using **`strptime`** (String Parse Time).

This is the opposite of `strftime`. It takes a string and a format, and gives you a datetime object.

In [ ]:
# Parsing Dates (`strptime`)
date = "26/02/2026"

new_date = dt.datetime.strptime(date, "%d/%m/%Y")
print(new_date)
# Since we didn't provide time, it defaults to 00:00:00
print(type(new_date))
print(new_date.year)
print(new_date.month)
print(new_date.hour)



In [ ]:
now = dt.datetime.utcnow()
print(now)

### `strftime` vs `strptime`

Confusion Alert: Students will mix these up.

`strFtime` = Format (Time -> String).

`strPtime` = Parse (String -> Time).

For beginners, focus 95% on `strftime`.

---
## ⏳ Topic 2: Time Travel (`timedelta`)
What if you want to know the date **7 days from now**?
You can't just do `today + 7` (Python doesn't know if you mean 7 days, 7 years, or 7 seconds).
We use `timedelta`.

In [ ]:
from datetime import timedelta

today = dt.datetime.now()
# adding timedelta of 7 days to today
one_week_later = today + timedelta(days=7)
# subtracting timedelta of 1 day from today
yesterday = today - timedelta(days=1)

print(f"Next Week: {one_week_later.strftime('%Y-%m-%d')}")
print(f"Yesterday: {yesterday.strftime('%Y-%m-%d')}")

# When you subtract two datetime objects, you get a timedelta object that represents the difference between them.
print(one_week_later - yesterday)
print(type(one_week_later - yesterday))

## 🌐 Topic 3: Datetime with Timezones
By default, `datetime` gives you "naive" datetime objects that don't know about timezones. This can lead to bugs when you work with people in different parts of the world.

To get basic understanding of what timezone means read [this answer from ChatGPT](https://chatgpt.com/canvas/shared/69a8261752408191aa766cb0b8975818)


#### 3 Ways to Handle Timezones in Python:
1. **Using `datetime`'s built-in timezone support** (Python 3.2+)

In [ ]:
# 1 Using timezones from datetime
from datetime import datetime, timezone

utc_time = datetime.now(timezone.utc)
print(utc_time)

In [ ]:
from datetime import datetime, timezone, timedelta

ist = timezone(timedelta(hours=5, minutes=30))

today = datetime.now(ist)
print(today)

2. **Using `pytz` library** (Third-party, more powerful, but more complex)

In [ ]:
# run this to install pytz
! pip install pytz

In [ ]:
# 2. using pytz library for timezones
import pytz
from datetime import datetime

# Get current time in UTC
utc_now = datetime.now(pytz.utc)
print(f"Current UTC Time: {utc_now}")

# Convert to US/Eastern timezone
eastern = pytz.timezone('US/Eastern')
eastern_now = utc_now.astimezone(eastern)
print(f"Current US/Eastern Time: {eastern_now}")

In [ ]:
# Creating datetime from explictly we need to call one extra step to localize it to the timezone. This is a common gotcha with pytz.
tz = pytz.timezone("Asia/Kolkata")
dt = tz.localize(datetime(2026, 3, 4, 10, 0))
print(dt)

**3. Using `zoneinfo` library for timezones (Python 3.9+) (Modern Way)**

In [ ]:
# 3. using zoneinfo library for timezones (Python 3.9+)
from datetime import datetime
from zoneinfo import ZoneInfo

dt = datetime.now(ZoneInfo("Asia/Kolkata"))
print(dt)

2026-03-05 11:55:01.728233Z

In [ ]:
# creating datetime from explicitly with zoneinfo is much easier than pytz

dt = datetime(2026, 3, 4, 10, 0, tzinfo=ZoneInfo("Asia/Kolkata"))
print(dt)
print(dt.astimezone(ZoneInfo("UTC")))

---
## 📂 Topic 4: File Automation (`os`, `pathlib` & `shutil`)
We already know `os.listdir()` and `os.mkdir()`.
Now let's learn to **Move** and **Delete**.

**Warning:** Python does not put deleted files in the Recycle Bin. **They are gone forever.**



### Setup: Create Dummy Files
Run this to create a mess we can clean up.

In [ ]:
import os

# Create a messy folder
os.makedirs("messy_folder", exist_ok=True)

# Create dummy text files
with open("messy_folder/note1.txt", "w") as f: f.write("Hi")
with open("messy_folder/note2.txt", "w") as f: f.write("Hi")
with open("messy_folder/image.png", "w") as f: f.write("Fake Image") # Just a text file pretending to be an image

print("✅ 'messy_folder' created with files.")

## 📦 Working With Paths using `pathlib`

pathlib is a modern way to handle file paths. It provides an object-oriented interface to the filesystem. It replaces old `os.path` usage.

In [ ]:
from pathlib import Path
# it does not matter if the file or folder exists or not. Path will handle it gracefully.
p = Path("messy_folder/report.csv")

print(p.name)
print(p.parent)
print(p.suffix)

**You can also check below for more methods of `pathlib.Path` objects:**

In [ ]:
print(p.exists()) # if path or file exists
print(p.is_file()) # if it's a file
print(p.is_dir()) # if it's a directory

**Create directories:**

In [ ]:
# Creates a directory if it doesn't exist, and does nothing if it already exists (thanks to `exist_ok=True`).
Path("messy_folder").mkdir(exist_ok=True)

**Open file:**

**NOTE**: The `/` operator is the neat trick from pathlib that joins paths safely across operating systems.

In [ ]:
# using `open` with `Path` objects is also possible, but using the `open` method of `Path` objects is more elegant and consistent with the object-oriented style of `pathlib`.
parent_path = Path("messy_folder") / "backup" 
parent_path.mkdir(parents=True, exist_ok=True)
file_path = parent_path / "data2.txt"


with file_path.open("w") as f:
    f.write("This is some log data.")

In [ ]:
# Open using pathlib's path and / operator to join paths
with open(Path("messy_folder") / "data2.txt", "w") as f:
    f.write("This is some log datasdfsfsfs.")

###

**Side Note: ** using `open` with `Path` objects is also possible, but using the `open` method of `Path` objects is more elegant and consistent with the object-oriented style of `pathlib`.

## 📦 Working with `shutil`
We use `shutil` (Shell Utilities) to copy, move, and delete files.

In [ ]:
import shutil

### Copying a file

In [ ]:
# copy a file to create backup
shutil.copy(Path("messy_folder") / "data1.txt", Path("messy_folder") / "data1_backup.txt")

### Moving a file

In [ ]:
# move a file from one directory to another directory

archive_path = Path("messy_folder") / "archive" / "data2.txt"
archive_path.mkdir(parents=True, exist_ok=True)
shutil.move(Path("messy_folder") / "data1.txt", archive_path / "data1.txt")

### Rename a file

In [ ]:
# rename using shutil
# move file under same directory to rename
shutil.move(Path("messy_folder") / "data2.txt", Path("messy_folder") / "data2_renamed.txt")

### The Danger of `shutil.move`

Warning: If you move a file to a folder that doesn't exist, `shutil` might rename the file to the folder's name (treating the folder path as a filename).

Fix: Always ensure the destination folder exists (`os.makedirs`) before moving files into it.

### Copy entire directory

In [ ]:
shutil.copytree("messy_folder", "messy_folder_backup")

### Delete an entire directory:

In [ ]:
shutil.rmtree("messy_folder")
shutil.rmtree("messy_folder_backup")

### When to use which

**Use pathlib when you are:**

- building file paths
- navigating directories
- checking existence
- reading or writing files

**Use shutil when you are:**

- copying files
- moving files
- deleting directory trees
- creating archives

---
## 🏋️ Day 6 Activities: The Automator

### Level 1: Time Travel (`timedelta`)
1. Import `datetime` and `timedelta`.
2. Get the current naive local time.
3. Calculate the date exactly 30 days and 12 hours from right now.
4. Print the future date.

In [ ]:
# Level 1 Code

### Level 2: Zone Aware (`zoneinfo` / `pytz`) 
1. Import `ZoneInfo` from `zoneinfo` (or `timezone` from `pytz`).
2. Create a timezone-aware datetime object for the current time in `"Asia/Tokyo"`.
3. Create another timezone-aware datetime for `"America/New_York"`.
4. Print both to see the time difference!

In [ ]:
# Level 2 Code

### Level 3: Modern Paths (`pathlib`) 📁
1. Import `Path` from `pathlib`.
2. Create a Path object for a new directory called `"project_files"`.
3. Use `.mkdir(exist_ok=True)` to create it.
4. Inside that directory, create a new Path object for `"log.txt"` and use `.touch()` to create the empty file.
5. Print whether the file `.exists()`.

In [ ]:
# Level 3 Code

### [MARKDOWN CELL]
### Level 4: The Mover (`shutil` + `pathlib`) 🚚
1. Import `shutil` and `Path`.
2. Create a Path for a new folder called `"backup"`. Make the folder.
3. Use `shutil.copy()` to copy the `"log.txt"` (from Level 3) into the `"backup"` folder.
4. Check if the backup file exists using Pathlib.

In [ ]:
# Level 4 Code

### Level 5: The "Old File" Cleaner (Advanced) 👴
**Scenario:** Clean up old logs.
1. Define a "cutoff date" (e.g., 30 days ago).
2. Compare it to a fake file date.
3. Logic:
   
   `file_date = datetime(2020, 1, 1)` (Very old)

   `if file_date < cutoff_date: print("Delete this file")`
   
   `else: print("Keep this file")`

In [ ]:
# Level 5 Code

### Fun Challange: Real World Use Case

Problem Statement: "Does your 'Downloads' folder has huge number of files like 5000?"

Task : "Your homework is to use Level 4 logic to build a script that moves all .pdfs to a PDF folder and .jpgs to a Pictures folder. You will feel like a wizard."